# 800 — External Bulk Cohorts

## PREFLIGHT-A — CPTAC-3 cohort eligibility and frozen-input readiness

**Status:** implementation draft; no biological replication analysis has been authorized or executed.

**Objective.** Convert the Notebook 111 CPTAC-3 acquisition audit into auditable, *technical* candidate inventories for prospective external-bulk validation of the three frozen Phase 4 consensus transcriptomic representations.

This initial Notebook 800 implementation is restricted to: (1) referencing the frozen acquisition evidence; (2) auditing primary-tumor candidate coverage and case multiplicity; (3) checking *lexical* overlap against the frozen TCGA sample mapping; (4) inspecting a deterministic, technical STAR Counts gene-ID pilot; and (5) writing non-frozen preflight diagnostics. It does **not** project consensus scores, construct a final analysis cohort, assess cross-dataset replication, test program concordance, or interpret methylation–expression evidence.

**Scientific boundaries.** Cohort eligibility, lineage crosswalk, donor independence, a one-record-per-case rule, full gene coverage, normalization, the replication estimand, null models and multiplicity handling must be prospectively reviewed and frozen before those analyses. Cohort source labels (`primary_site`, `disease_type`) must not be silently converted into TCGA lineages. CPTAC-3 RNA-seq eligibility must not be conditioned on DNA-methylation availability.

**Sources.** `111_cptac3_acquisition_and_audit.ipynb`, `docs/contracts/phase8/NOTEBOOK_800_PREFLIGHT_A_DRAFT.md`, `docs/NOTEBOOK_STYLE_GUIDE.md`, `config/artifact_registry.json` and `config/raw_data_registry.json`.

Run from the project's **Python 3.11.8** Jupyter environment with the local Notebook 111 outputs already generated. No GDC calls, re-download, upstream re-execution or full raw payload MD5 scan is performed.

In [1]:
# =============================================================================
# Imports
# =============================================================================

import json
import platform

import numpy as np
import pandas as pd

from pancancer_epigenetics.utils.artifact_registry import (
    get_artifact,
    load_artifact_registry,
    resolve_artifact_path,
)
from pancancer_epigenetics.utils.file_checks import calculate_sha256
from pancancer_epigenetics.utils.paths import Paths, project_relative_path
from pancancer_epigenetics.utils.raw_data_registry import load_raw_data_registry

In [2]:
# =============================================================================
# Notebook 111 handoff and stable Phase 4 input identifiers
# =============================================================================

AUDIT_SUMMARY_PATH = Paths.metadata / "111_cptac3_acquisition_audit_summary.json"
RAW_REGISTRY_PROPOSAL_PATH = Paths.metadata / "111_cptac3_raw_registry_proposal.json"
OUTPUT_DIR = Paths.metadata

AUDIT_INPUT_KEYS = (
    "file_index",
    "candidate_primary",
    "ambiguity_by_lineage",
)

FROZEN_ARTIFACT_IDS = {
    "gene_universe": "phase4.400.cross_system_shared_gene_universe",
    "program_catalog": "phase4.401.consensus_transcriptomic_program_catalog",
    "program_weights": "phase4.401.consensus_transcriptomic_gene_weights",
    "construction_metadata": "phase4.401.consensus_construction_metadata",
    "tcga_sample_mapping": "phase2.203.final_sample_mapping",
}

OUTPUT_PATHS = {
    "lineage_inventory": OUTPUT_DIR / "800_preflight_a_candidate_lineage_inventory.csv",
    "case_multiplicity": OUTPUT_DIR / "800_preflight_a_case_multiplicity.csv",
    "identifier_collisions": OUTPUT_DIR / "800_preflight_a_identifier_collisions.csv",
    "gene_id_pilot": OUTPUT_DIR / "800_preflight_a_gene_id_pilot.csv",
    "readiness": OUTPUT_DIR / "800_preflight_a_readiness.json",
    "rna_adjudication_case_evidence": OUTPUT_DIR / "800_preflight_b_rna_case_evidence.csv",
    "rna_adjudication_file_evidence": OUTPUT_DIR / "800_preflight_b_rna_file_evidence.csv",
    "rna_c_field_coverage": OUTPUT_DIR / "800_preflight_c_rna_field_coverage.csv",
    "rna_c_file_availability": OUTPUT_DIR / "800_preflight_c_rna_file_availability.csv",
    "rna_c_case_coverage": OUTPUT_DIR / "800_preflight_c_rna_case_coverage.csv",
}

In [3]:
# =============================================================================
# Read Notebook 111's observed audit boundary
# =============================================================================

with AUDIT_SUMMARY_PATH.open("r", encoding="utf-8") as handle:
    audit_summary = json.load(handle)

if audit_summary.get("notebook") != 111:
    raise ValueError("The source is not a Notebook 111 audit summary")
if audit_summary.get("cohort_frozen") is not False:
    raise ValueError("Unexpected cohort-freeze flag in Notebook 111 handoff")
if audit_summary.get("phase8_validation_performed") is not False:
    raise ValueError("Unexpected Phase 8 validation status")
if audit_summary.get("n_files_by_modality") != {
    "rnaseq": 2968, "methylation": 2610,
}:
    raise ValueError("CPTAC-3 source-snapshot identity differs from Notebook 111")

print("Notebook 111 source audit: PASS (reported)")
print("Cohort freeze:", audit_summary["cohort_frozen"])
print("External validation performed:", audit_summary["phase8_validation_performed"])

Notebook 111 source audit: PASS (reported)
Cohort freeze: False
External validation performed: False


In [4]:
# =============================================================================
# Resolve and fingerprint the small derived audit tables consumed by 800
# =============================================================================

audit_inputs = {}
for name in AUDIT_INPUT_KEYS:
    record = audit_summary["audit_artifacts"][name]
    path = Paths.root / record["path"]
    if not path.is_file():
        raise FileNotFoundError(f"Missing Notebook 111 derived handoff: {path}")
    if calculate_sha256(path) != record["sha256"]:
        raise ValueError(f"Notebook 111 derived handoff changed: {name}")
    audit_inputs[name] = path

print("Verified Notebook 111 derived handoffs:")
for name, path in audit_inputs.items():
    print(" ", name, project_relative_path(path))

Verified Notebook 111 derived handoffs:
  file_index data/interim/metadata/111_cptac3_file_index.csv
  candidate_primary data/interim/metadata/111_cptac3_provisional_primary_tumor_files.csv
  ambiguity_by_lineage data/interim/metadata/111_cptac3_ambiguity_by_lineage.csv


In [5]:
# =============================================================================
# Read authoritative Notebook 111 tables for new downstream questions only
# =============================================================================

file_index = pd.read_csv(audit_inputs["file_index"], dtype="string")
candidate_primary = pd.read_csv(audit_inputs["candidate_primary"], dtype="string")
upstream_ambiguity = pd.read_csv(
    audit_inputs["ambiguity_by_lineage"], dtype="string"
)

new_use_columns = {
    "modality", "file_id", "filename", "case_uuid", "case_submitter_id",
    "sample_uuid", "aliquot_uuid", "mapping_status",
    "primary_site", "disease_type", "platform",
}
missing_columns = new_use_columns - set(file_index.columns)
if missing_columns:
    raise KeyError(f"Notebook 800 required columns missing: {sorted(missing_columns)}")

# Notebook 111's mapping-status distribution, source totals and primary
# sample-type inventory are authoritative and are not recalculated here.

## Source-registration boundary

The Notebook 111 registry proposal is a review artifact, **not** permission to silently modify the raw-data registry. A missing `cptac3` entry must remain visible as a registration task. Likewise, Phase 4 inputs are frozen and consumed only after exact identity checks; their weights are *not* applied to CPTAC-3 in this preflight.

In [6]:
# =============================================================================
# Inspect CPTAC-3 raw registry state without mutating it
# =============================================================================

raw_registry = load_raw_data_registry()
with RAW_REGISTRY_PROPOSAL_PATH.open("r", encoding="utf-8") as handle:
    cptac3_registry_proposal = json.load(handle)

cptac3_registered = "cptac3" in raw_registry
print("CPTAC-3 registered:", cptac3_registered)
print("Proposed RNA status:",
      cptac3_registry_proposal["cohorts"]["rnaseq"]["status"])
print("Proposed methylation status:",
      cptac3_registry_proposal["cohorts"]["methylation"]["status"])
if cptac3_registered:
    print("Existing registration requires review against Notebook 111 proposal")

CPTAC-3 registered: False
Proposed RNA status: acquired_not_used
Proposed methylation status: acquired_not_used


In [7]:
# =============================================================================
# Verify exact frozen Phase 4 and TCGA reference identities
# =============================================================================

artifact_registry = load_artifact_registry()
frozen_paths = {}
for name, artifact_id in FROZEN_ARTIFACT_IDS.items():
    spec = get_artifact(artifact_registry, artifact_id)
    path = resolve_artifact_path(artifact_registry, artifact_id)
    if spec["status"] != "frozen":
        raise ValueError(f"Required upstream artifact is not frozen: {artifact_id}")
    if path.stat().st_size != spec["size_bytes"]:
        raise ValueError(f"Frozen upstream file size changed: {artifact_id}")
    if calculate_sha256(path) != spec["sha256"]:
        raise ValueError(f"Frozen upstream fingerprint changed: {artifact_id}")
    frozen_paths[name] = path

print("Verified", len(frozen_paths), "frozen input identities")

Verified 5 frozen input identities


In [8]:
# =============================================================================
# Read the frozen gene universe and consensus metadata without scoring
# =============================================================================

shared_genes = pd.read_csv(frozen_paths["gene_universe"], dtype="string")
program_catalog = pd.read_csv(frozen_paths["program_catalog"], dtype="string")
program_weights = pd.read_csv(frozen_paths["program_weights"], dtype="string")
with frozen_paths["construction_metadata"].open("r", encoding="utf-8") as handle:
    construction_metadata = json.load(handle)

required_gene_columns = {"gene_symbol", "tumor_gene_id", "cell_line_gene_id"}
if required_gene_columns - set(shared_genes.columns):
    raise KeyError("Frozen gene universe lacks identifiers required for a technical crosswalk")
if {"consensus_program_id", "gene_symbol", "consensus_weight"} - set(program_weights):
    raise KeyError("Frozen gene weights lack identity columns")

print("Frozen genes:", len(shared_genes))
print("Program IDs:", sorted(program_weights["consensus_program_id"].unique().tolist()))
print("Frozen weight rows:", len(program_weights))
print("Score standardization:", construction_metadata["score_construction"])

Frozen genes: 2389
Program IDs: ['CONSENSUS_TX_01', 'CONSENSUS_TX_02', 'CONSENSUS_TX_03']
Frozen weight rows: 7167
Score standardization: {'gene_standardization': 'within_system_mean0_sd1', 'projection': 'standardized_expression_dot_consensus_loading', 'final_score_standardization': 'within_system_per_program_mean0_sd1', 'cross_system_joint_normalization': False, 'absolute_score_scale_cross_system_comparable': False}


## Lineage-aware review of **new** case-level eligibility risks

Notebook 111 already froze technical evidence on sample ancestry, including an
`ambiguity_by_lineage` table with **file-level** denominators and a provisional
primary-tumor inventory. Notebook 800 consumes those audited numbers directly.

The new question is case-level: how many *distinct cases* in each existing
GDC stratum are touched by unresolved file ancestry, and how does that compare
descriptively with the pool of uniquely mapped provisional primary-tumor cases?
These are **not** new primary-tumor calls, new lineage mappings, or eligibility
criteria. The denominator of the unresolved-case fraction contains **all sample
types** from the acquired files, and cannot be interpreted as primary-tumor
attrition. A dedicated primary-compatible unresolved-sample sensitivity, and
the final one-case selection rule, remain separate decisions.

In [9]:
# =============================================================================
# Reuse Notebook 111 source-stratum counts; compute only new case-level risk
# =============================================================================

STRATUM_KEYS = ["modality", "primary_site", "disease_type", "platform"]
UPSTREAM_COUNT_COLUMNS = [
    "n_files", "n_cases", "n_ambiguous_files", "ambiguous_file_fraction",
]
missing_upstream = (
    set(STRATUM_KEYS + UPSTREAM_COUNT_COLUMNS) - set(upstream_ambiguity.columns)
)
if missing_upstream:
    raise KeyError(f"111 lineage audit missing fields: {sorted(missing_upstream)}")

source_counts = upstream_ambiguity[
    STRATUM_KEYS + UPSTREAM_COUNT_COLUMNS
].copy()
for column in UPSTREAM_COUNT_COLUMNS:
    source_counts[column] = pd.to_numeric(
        source_counts[column], errors="raise"
    )

unresolved_case_counts = (
    file_index.loc[file_index["mapping_status"].eq("ambiguous_sample_ancestry")]
    .groupby(STRATUM_KEYS, dropna=False)["case_uuid"]
    .nunique()
    .rename("cases_with_ambiguous_files")
    .reset_index()
)
source_counts = source_counts.merge(
    unresolved_case_counts, on=STRATUM_KEYS, how="left", validate="one_to_one"
)
source_counts["cases_with_ambiguous_files"] = (
    source_counts["cases_with_ambiguous_files"].fillna(0).astype("int64")
)
source_counts["fraction_all_cases_with_ambiguous_files"] = (
    source_counts["cases_with_ambiguous_files"] / source_counts["n_cases"]
)

In [10]:
# =============================================================================
# Contextualize new case-level ambiguity against the 111 primary inventory
# =============================================================================

# Notebook 111 reports primary counts by source site/disease; this new
# platform-specific case view is descriptive only and is not an eligibility gate.
primary_case_counts = (
    candidate_primary.groupby(STRATUM_KEYS, dropna=False)["case_uuid"]
    .nunique()
    .rename("provisional_primary_cases")
    .reset_index()
)

lineage_inventory = source_counts.merge(
    primary_case_counts, on=STRATUM_KEYS, how="left", validate="one_to_one"
)
lineage_inventory["provisional_primary_cases"] = (
    lineage_inventory["provisional_primary_cases"].fillna(0).astype("int64")
)
lineage_inventory = lineage_inventory.sort_values(
    ["modality", "cases_with_ambiguous_files", "primary_site", "disease_type"],
    ascending=[True, False, True, True], kind="stable"
).reset_index(drop=True)

display(lineage_inventory[
    ["modality", "primary_site", "disease_type", "platform",
     "n_cases", "cases_with_ambiguous_files",
     "fraction_all_cases_with_ambiguous_files", "provisional_primary_cases"]
].head(25))
print("Lineage crosswalk and primary-specific attrition: NOT ASSESSED")

,modality,primary_site,disease_type,platform,n_cases,cases_with_ambiguous_files,fraction_all_cases_with_ambiguous_files,provisional_primary_cases
0,methylation,Pancreas,Ductal and Lobular Neoplasms,Illumina Methylation Epic,146,62,0.424658,83
1,methylation,Bronchus and lung,Adenomas and Adenocarcinomas,Illumina Methylation Epic,230,58,0.252174,177
2,methylation,Brain,Gliomas,Illumina Methylation Epic,205,38,0.185366,161
3,methylation,Kidney,Adenomas and Adenocarcinomas,Illumina Methylation Epic,256,27,0.105469,231
4,methylation,Skin,Nevi and Melanomas,Illumina Methylation Epic,159,24,0.150943,134
5,methylation,Bronchus and lung,Squamous Cell Neoplasms,Illumina Methylation Epic,109,21,0.192661,89
6,methylation,"Uterus, NOS",Adenomas and Adenocarcinomas,Illumina Methylation Epic,241,20,0.082988,220
7,methylation,Larynx,Squamous Cell Neoplasms,Illumina Methylation Epic,49,15,0.306122,35
8,methylation,Stomach,Adenomas and Adenocarcinomas,Illumina Methylation Epic v2,164,11,0.067073,151
9,methylation,Skin,Nevi and Melanomas,Illumina Methylation Epic v2,18,6,0.333333,1


Lineage crosswalk and primary-specific attrition: NOT ASSESSED


In [11]:
# =============================================================================
# Quantify case-level multiplicity in provisional primary records
# =============================================================================

case_multiplicity = (
    candidate_primary.groupby(["modality", "case_uuid"], dropna=False)
    .agg(
        n_files=("file_id", "nunique"),
        n_samples=("sample_uuid", "nunique"),
        n_aliquots=("aliquot_uuid", "nunique"),
        n_platforms=("platform", "nunique"),
        file_uuid_values=("file_id", lambda v: "|".join(sorted(set(v.dropna())))),
        sample_uuid_values=("sample_uuid", lambda v: "|".join(sorted(set(v.dropna())))),
        aliquot_uuid_values=("aliquot_uuid", lambda v: "|".join(sorted(set(v.dropna())))),
        primary_site_values=("primary_site", lambda v: "|".join(sorted(set(v.dropna())))),
        disease_type_values=("disease_type", lambda v: "|".join(sorted(set(v.dropna())))),
    )
    .reset_index()
)

case_multiplicity["needs_selection_policy"] = (
    case_multiplicity[["n_files", "n_samples", "n_aliquots"]].gt(1).any(axis=1)
)
case_multiplicity["selected_file_id"] = pd.Series(
    pd.NA, index=case_multiplicity.index, dtype="string"
)
case_multiplicity["selection_status"] = "NOT_FROZEN"
case_multiplicity = case_multiplicity.sort_values(
    ["modality", "case_uuid"], kind="stable"
).reset_index(drop=True)

display(
    case_multiplicity.groupby("modality")
    .agg(
        candidate_cases=("case_uuid", "nunique"),
        cases_needing_review=("needs_selection_policy", "sum"),
        maximum_files_per_case=("n_files", "max"),
        maximum_samples_per_case=("n_samples", "max"),
    )
)
print("No case-level selection, averaging or technical-replicate pooling performed")

,candidate_cases,cases_needing_review,maximum_files_per_case,maximum_samples_per_case
modality,,,,
methylation,1336,62,5,5
rnaseq,1370,288,8,5


No case-level selection, averaging or technical-replicate pooling performed


## RNA-seq per-case multiplicity: outcome-blind structural classification

The initial 800 preflight identified **288 provisional RNA-seq cases requiring multiplicity review**. The next technical task is to distinguish case-level identifier structures **without** reopening Notebook 111's file-to-sample ancestry audit or making a biological sample-selection decision.

Classification is based only on already audited file, sample, and aliquot UUIDs. **More than one file on a single aliquot is not, by itself, proof of technical replication or equivalence of processed files.** More than one aliquot under one sample is not automatically exchangeable, and distinct sample UUIDs under one case cannot be pooled as independent patients.

Cases with internally inconsistent counts or cross-file mapping conflicts must remain explicitly unresolved. Even a case with one file is merely *technically simple*, **not** finally eligible: donor independence, lineage harmonization, technical QC and prospective Phase 8 scoring policy remain open.

In [12]:
# =============================================================================
# Classify RNA-seq case multiplicity from the existing 800 case inventory
# =============================================================================

rna_case_mask = case_multiplicity["modality"].eq("rnaseq")
rna_cases = case_multiplicity.loc[rna_case_mask].copy()

inconsistent_counts = (
    rna_cases[["n_files", "n_samples", "n_aliquots"]].le(0).any(axis=1)
    | rna_cases["n_aliquots"].gt(rna_cases["n_files"])
    | rna_cases["n_samples"].gt(rna_cases["n_aliquots"])
)
rna_cases["rna_multiplicity_class"] = np.select(
    [
        inconsistent_counts,
        rna_cases["n_samples"].gt(1),
        rna_cases["n_aliquots"].gt(1),
        rna_cases["n_files"].gt(1),
    ],
    [
        "INCONSISTENT_BIOSPECIMEN_COUNTS",
        "MULTIPLE_SAMPLES_IN_CASE",
        "ONE_SAMPLE_MULTIPLE_ALIQUOTS",
        "ONE_ALIQUOT_MULTIPLE_FILES",
    ],
    default="SINGLE_FILE_SAMPLE_ALIQUOT",
)
print("RNA-seq provisional cases classified:", len(rna_cases))
print("No record selected, pooled, excluded or scored")

RNA-seq provisional cases classified: 1370
No record selected, pooled, excluded or scored


In [13]:
# =============================================================================
# Screen for cross-file ancestry conflicts not resolved by single-file links
# =============================================================================

rna_candidate_rows = candidate_primary.loc[
    candidate_primary["modality"].eq("rnaseq"),
    ["case_uuid", "sample_uuid", "aliquot_uuid", "file_id"],
].copy()

aliquot_links = (
    rna_candidate_rows.groupby("aliquot_uuid")
    .agg(
        n_parent_samples=("sample_uuid", "nunique"),
        n_parent_cases=("case_uuid", "nunique"),
    )
)
sample_links = (
    rna_candidate_rows.groupby("sample_uuid")
    .agg(n_parent_cases=("case_uuid", "nunique"))
)
conflicting_aliquots = set(aliquot_links.loc[
    aliquot_links["n_parent_samples"].gt(1)
    | aliquot_links["n_parent_cases"].gt(1)
].index)
conflicting_samples = set(sample_links.loc[
    sample_links["n_parent_cases"].gt(1)
].index)

rna_cases["cross_file_ancestry_conflict"] = (
    rna_cases["aliquot_uuid_values"].map(
        lambda s: any(part in conflicting_aliquots for part in s.split("|"))
    )
    | rna_cases["sample_uuid_values"].map(
        lambda s: any(part in conflicting_samples for part in s.split("|"))
    )
)
rna_cases.loc[
    rna_cases["cross_file_ancestry_conflict"],
    "rna_multiplicity_class",
] = "CROSS_FILE_BIOSPECIMEN_CONFLICT"

print("Aliquot UUIDs with incompatible cross-file parents:", len(conflicting_aliquots))
print("Sample UUIDs with incompatible cross-file cases:", len(conflicting_samples))
print("Affected RNA cases:", int(rna_cases["cross_file_ancestry_conflict"].sum()))

Aliquot UUIDs with incompatible cross-file parents: 0
Sample UUIDs with incompatible cross-file cases: 0
Affected RNA cases: 0


In [14]:
# =============================================================================
# Show nonoverlapping RNA-seq structural classes and review candidates
# =============================================================================

rna_class_summary = (
    rna_cases.groupby("rna_multiplicity_class", dropna=False)
    .agg(
        n_cases=("case_uuid", "nunique"),
        max_files_in_case=("n_files", "max"),
        max_samples_in_case=("n_samples", "max"),
    )
    .reset_index()
    .sort_values(["n_cases", "rna_multiplicity_class"], ascending=[False, True])
)
rna_class_summary["fraction_provisional_rna_cases"] = (
    rna_class_summary["n_cases"] / len(rna_cases)
)

display(rna_class_summary)
print(
    "Cases previously flagged for multiplicity review:",
    int(rna_cases["needs_selection_policy"].sum()),
)
print("Unresolved case selection: ALL — no file selection rule frozen")

# Preserve exact source labels; mixed labels must remain visible.
display(
    rna_cases.loc[rna_cases["needs_selection_policy"]]
    .groupby(["primary_site_values", "rna_multiplicity_class"], dropna=False)
    .size().rename("n_cases").reset_index()
    .sort_values(["n_cases", "primary_site_values"], ascending=[False, True])
    .head(20)
)

,rna_multiplicity_class,n_cases,max_files_in_case,max_samples_in_case,fraction_provisional_rna_cases
2,SINGLE_FILE_SAMPLE_ALIQUOT,1082,1,1,0.789781
0,MULTIPLE_SAMPLES_IN_CASE,276,8,5,0.201460
1,ONE_SAMPLE_MULTIPLE_ALIQUOTS,12,3,1,0.008759


Cases previously flagged for multiplicity review: 288
Unresolved case selection: ALL — no file selection rule frozen


,primary_site_values,rna_multiplicity_class,n_cases
13,"Uterus, NOS",MULTIPLE_SAMPLES_IN_CASE,179
4,Kidney,MULTIPLE_SAMPLES_IN_CASE,40
5,Larynx,MULTIPLE_SAMPLES_IN_CASE,19
11,Other and unspecified parts of tongue,MULTIPLE_SAMPLES_IN_CASE,12
12,Pancreas,ONE_SAMPLE_MULTIPLE_ALIQUOTS,12
2,Floor of mouth,MULTIPLE_SAMPLES_IN_CASE,11
3,Gum,MULTIPLE_SAMPLES_IN_CASE,3
7,Oropharynx,MULTIPLE_SAMPLES_IN_CASE,3
9,"Other and ill-defined sites in lip, oral cavit...",MULTIPLE_SAMPLES_IN_CASE,3
6,Lip,MULTIPLE_SAMPLES_IN_CASE,2


In [15]:
# =============================================================================
# Annotate the existing case-level output; create no duplicate source inventory
# =============================================================================

case_multiplicity["rna_multiplicity_class"] = pd.Series(
    pd.NA, index=case_multiplicity.index, dtype="string"
)
case_multiplicity["rna_cross_file_ancestry_conflict"] = pd.Series(
    pd.NA, index=case_multiplicity.index, dtype="boolean"
)

case_multiplicity.loc[
    rna_case_mask, "rna_multiplicity_class"
] = rna_cases["rna_multiplicity_class"].to_numpy()
case_multiplicity.loc[
    rna_case_mask, "rna_cross_file_ancestry_conflict"
] = rna_cases["cross_file_ancestry_conflict"].to_numpy()

if (
    rna_class_summary["n_cases"].sum() != len(rna_cases)
    or case_multiplicity.loc[rna_case_mask, "rna_multiplicity_class"].isna().any()
):
    raise ValueError("Incomplete RNA-seq case multiplicity classification")

print("Annotated output:", project_relative_path(OUTPUT_PATHS["case_multiplicity"]))
print("Classification: descriptive only; no sample selected")

Annotated output: data/interim/metadata/800_preflight_a_case_multiplicity.csv
Classification: descriptive only; no sample selected


## PREFLIGHT-B — Outcome-blind RNA-seq multiplicity adjudication evidence

**Question:** for the 288 cases containing multiple uniquely mapped provisional primary-tumor RNA-seq files/samples/aliquots, which *existing, audited* source metadata can distinguish candidate biospecimens? Does that information supply a defensible basis for selecting one observation per case?

This is a **metadata evidence audit**, not an actual adjudication. It consumes the already classified RNA cases and Notebook 111's frozen candidate file index; it does not rerun source acquisition, reopen ancestral assignment, reinterpret ambiguous files, or inspect any gene expression or program scores.

The preserved GDC descriptors `sample_type`, `tissue_type` and `tumor_descriptor` may identify disagreements but cannot resolve tumor-region/timepoint differences by themselves. Sample/aliquot submitter IDs are identifiers, not evidence of relative quality. Provider file size, MD5 and generic STAR workflow are **not sequencing QC selection metrics**. Even identical submitted labels do not prove samples are interchangeable.

All 288 case/file identities remain in a **NOT_FROZEN** state. Any future one-per-case decision must use a separately reviewed outcome-blind policy, with enough objective specimen provenance and technical QC. Cases without a defensible selection basis remain unresolved; never select on UUID ordering, file size, gene expression or program score.

In [16]:
# =============================================================================
# Restrict new evidence review to the 288 RNA cases already identified in 800
# =============================================================================

rna_review_cases = rna_cases.loc[
    rna_cases["needs_selection_policy"], [
        "case_uuid", "rna_multiplicity_class", "n_files", "n_samples",
        "n_aliquots", "cross_file_ancestry_conflict",
    ]
].copy()

RNA_SOURCE_EVIDENCE_COLS = [
    "case_uuid", "file_id", "filename", "sample_uuid", "sample_submitter_id",
    "aliquot_uuid", "aliquot_submitter_id", "sample_type", "tissue_type",
    "tumor_descriptor", "primary_site", "disease_type", "project_id",
    "workflow_type", "platform", "size_bytes", "provider_md5",
]
missing = set(RNA_SOURCE_EVIDENCE_COLS) - set(candidate_primary.columns)
if missing:
    raise KeyError("Required existing Notebook 111 metadata absent: " + str(sorted(missing)))

rna_adjudication_files = candidate_primary.loc[
    candidate_primary["modality"].eq("rnaseq")
    & candidate_primary["case_uuid"].isin(rna_review_cases["case_uuid"]),
    RNA_SOURCE_EVIDENCE_COLS,
].copy()

if (
    rna_review_cases["case_uuid"].duplicated().any()
    or rna_adjudication_files["file_id"].duplicated().any()
    or set(rna_adjudication_files["case_uuid"]) != set(rna_review_cases["case_uuid"])
    or len(rna_adjudication_files) != int(rna_review_cases["n_files"].sum())
):
    raise ValueError("800-B candidate review does not reconcile to 800-A case inventory")

print("Cases under metadata adjudication review:", len(rna_review_cases))
print("Existing primary RNA file records under review:", len(rna_adjudication_files))

Cases under metadata adjudication review: 288
Existing primary RNA file records under review: 649


In [17]:
# =============================================================================
# Quantify per-case specimen descriptors and source-file metadata coverage
# =============================================================================

def nonnull_count(series):
    return int(series.notna().sum())

def unique_nonnull_count(series):
    return int(series.nunique(dropna=True))

evidence = (
    rna_adjudication_files.groupby("case_uuid", dropna=False)
    .agg(
        observed_files=("file_id", "nunique"),
        observed_samples=("sample_uuid", "nunique"),
        observed_aliquots=("aliquot_uuid", "nunique"),
        sample_submitter_ids=("sample_submitter_id", unique_nonnull_count),
        sample_submitter_id_file_coverage=("sample_submitter_id", nonnull_count),
        aliquot_submitter_ids=("aliquot_submitter_id", unique_nonnull_count),
        aliquot_submitter_id_file_coverage=("aliquot_submitter_id", nonnull_count),
        tissue_type_values=("tissue_type", unique_nonnull_count),
        tissue_type_file_coverage=("tissue_type", nonnull_count),
        tumor_descriptor_values=("tumor_descriptor", unique_nonnull_count),
        tumor_descriptor_file_coverage=("tumor_descriptor", nonnull_count),
        source_site_values=("primary_site", unique_nonnull_count),
        source_disease_values=("disease_type", unique_nonnull_count),
        project_values=("project_id", unique_nonnull_count),
        workflow_values=("workflow_type", unique_nonnull_count),
        platform_values=("platform", unique_nonnull_count),
        distinct_provider_md5=("provider_md5", unique_nonnull_count),
    ).reset_index()
)
rna_adjudication_cases = rna_review_cases.merge(
    evidence, on="case_uuid", how="left", validate="one_to_one"
)

if not (
    rna_adjudication_cases["n_files"].eq(rna_adjudication_cases["observed_files"]).all()
    and rna_adjudication_cases["n_samples"].eq(
        rna_adjudication_cases["observed_samples"]
    ).all()
    and rna_adjudication_cases["n_aliquots"].eq(
        rna_adjudication_cases["observed_aliquots"]
    ).all()
):
    raise ValueError("New specimen evidence does not match prior case structure")

display(
    rna_adjudication_cases[[
        "rna_multiplicity_class", "observed_files",
        "tissue_type_file_coverage", "tumor_descriptor_file_coverage",
        "sample_submitter_id_file_coverage",
        "aliquot_submitter_id_file_coverage",
    ]].groupby("rna_multiplicity_class").sum()
)

,observed_files,tissue_type_file_coverage,tumor_descriptor_file_coverage,sample_submitter_id_file_coverage,aliquot_submitter_id_file_coverage
rna_multiplicity_class,,,,,
MULTIPLE_SAMPLES_IN_CASE,619,619,619,619,619
ONE_SAMPLE_MULTIPLE_ALIQUOTS,30,30,30,30,30


In [18]:
# =============================================================================
# Flag metadata discordance and insufficiency, without resolving samples
# =============================================================================

rna_adjudication_cases["source_descriptor_discordance"] = (
    rna_adjudication_cases[[
        "tissue_type_values", "tumor_descriptor_values",
        "source_site_values", "source_disease_values",
        "project_values", "workflow_values", "platform_values",
    ]].gt(1).any(axis=1)
)
rna_adjudication_cases["source_descriptor_incomplete"] = (
    rna_adjudication_cases["tissue_type_file_coverage"].lt(
        rna_adjudication_cases["n_files"]
    )
    | rna_adjudication_cases["tumor_descriptor_file_coverage"].lt(
        rna_adjudication_cases["n_files"]
    )
    | rna_adjudication_cases["sample_submitter_id_file_coverage"].lt(
        rna_adjudication_cases["n_files"]
    )
    | rna_adjudication_cases["aliquot_submitter_id_file_coverage"].lt(
        rna_adjudication_cases["n_files"]
    )
)
rna_adjudication_cases["same_md5_across_distinct_files"] = (
    rna_adjudication_cases["distinct_provider_md5"].lt(
        rna_adjudication_cases["n_files"]
    )
)

rna_adjudication_cases["adjudication_evidence_class"] = np.select(
    [
        rna_adjudication_cases["cross_file_ancestry_conflict"],
        rna_adjudication_cases["source_descriptor_discordance"],
        rna_adjudication_cases["source_descriptor_incomplete"],
        rna_adjudication_cases["rna_multiplicity_class"].eq(
            "MULTIPLE_SAMPLES_IN_CASE"
        ),
    ],
    [
        "CROSS_FILE_CONFLICT_UNRESOLVED",
        "SOURCE_DESCRIPTORS_DISCORDANT_REVIEW",
        "SOURCE_DESCRIPTORS_INCOMPLETE_REVIEW",
        "MULTIPLE_SAMPLES_NO_QC_SELECTION_BASIS",
    ],
    default="MULTIPLE_ALIQUOTS_OR_FILES_NO_QC_SELECTION_BASIS",
)
rna_adjudication_cases["selection_evidence_status"] = (
    "INSUFFICIENT_TO_SELECT_WITHOUT_PROSPECTIVE_QC_POLICY"
)
rna_adjudication_cases["selected_file_id"] = pd.Series(
    pd.NA, index=rna_adjudication_cases.index, dtype="string"
)

display(
    rna_adjudication_cases["adjudication_evidence_class"]
    .value_counts(dropna=False)
    .rename_axis("evidence_class").reset_index(name="n_cases")
)
print("No files selected or excluded; no technical QC read from STAR payloads")

,evidence_class,n_cases
0,MULTIPLE_SAMPLES_NO_QC_SELECTION_BASIS,276
1,MULTIPLE_ALIQUOTS_OR_FILES_NO_QC_SELECTION_BASIS,12


No files selected or excluded; no technical QC read from STAR payloads


In [19]:
# =============================================================================
# Describe the evidence limits by source tumor context without rerunning 111
# =============================================================================

rna_case_context = rna_cases.loc[
    rna_cases["needs_selection_policy"],
    ["case_uuid", "primary_site_values", "disease_type_values"],
].copy()
rna_adjudication_cases = rna_adjudication_cases.merge(
    rna_case_context, on="case_uuid", validate="one_to_one"
)
display(
    rna_adjudication_cases.groupby(
        ["primary_site_values", "rna_multiplicity_class"], dropna=False
    ).agg(
        n_cases=("case_uuid", "nunique"),
        descriptor_discordance=("source_descriptor_discordance", "sum"),
        incomplete_descriptors=("source_descriptor_incomplete", "sum"),
        same_md5_file_groups=("same_md5_across_distinct_files", "sum"),
    )
    .reset_index()
    .sort_values(["n_cases", "primary_site_values"], ascending=[False, True])
    .head(25)
)
print(
    "Cases with source-descriptor disagreement:",
    int(rna_adjudication_cases["source_descriptor_discordance"].sum())
)
print(
    "Cases with at least one missing specimen descriptor field:",
    int(rna_adjudication_cases["source_descriptor_incomplete"].sum())
)
print("True technical replicate identity / relative RNA-seq QC: NOT ASSESSED")

,primary_site_values,rna_multiplicity_class,n_cases,descriptor_discordance,incomplete_descriptors,same_md5_file_groups
13,"Uterus, NOS",MULTIPLE_SAMPLES_IN_CASE,179,0,0,0
4,Kidney,MULTIPLE_SAMPLES_IN_CASE,40,0,0,0
5,Larynx,MULTIPLE_SAMPLES_IN_CASE,19,0,0,0
11,Other and unspecified parts of tongue,MULTIPLE_SAMPLES_IN_CASE,12,0,0,0
12,Pancreas,ONE_SAMPLE_MULTIPLE_ALIQUOTS,12,0,0,0
2,Floor of mouth,MULTIPLE_SAMPLES_IN_CASE,11,0,0,0
3,Gum,MULTIPLE_SAMPLES_IN_CASE,3,0,0,0
7,Oropharynx,MULTIPLE_SAMPLES_IN_CASE,3,0,0,0
9,"Other and ill-defined sites in lip, oral cavit...",MULTIPLE_SAMPLES_IN_CASE,3,0,0,0
6,Lip,MULTIPLE_SAMPLES_IN_CASE,2,0,0,0


Cases with source-descriptor disagreement: 0
Cases with at least one missing specimen descriptor field: 0
True technical replicate identity / relative RNA-seq QC: NOT ASSESSED


In [20]:
# =============================================================================
# Prepare auditable per-file metadata evidence; preserve case status UNRESOLVED
# =============================================================================

rna_adjudication_files = rna_adjudication_files.merge(
    rna_adjudication_cases[
        ["case_uuid", "rna_multiplicity_class", "adjudication_evidence_class",
         "selection_evidence_status"]
    ],
    on="case_uuid", validate="many_to_one"
)
rna_adjudication_files["candidate_selected"] = False
rna_adjudication_files["selection_performed"] = False

rna_adjudication_cases = rna_adjudication_cases.sort_values(
    "case_uuid", kind="stable"
).reset_index(drop=True)
rna_adjudication_files = rna_adjudication_files.sort_values(
    ["case_uuid", "sample_uuid", "aliquot_uuid", "file_id"], kind="stable"
).reset_index(drop=True)

if (
    len(rna_adjudication_cases) != int(rna_cases["needs_selection_policy"].sum())
    or rna_adjudication_cases["selected_file_id"].notna().any()
    or rna_adjudication_files["candidate_selected"].any()
):
    raise ValueError("Evidence-only 800-B unexpectedly selected or lost cases")

print("Case-level adjudication evidence rows:", len(rna_adjudication_cases))
print("File-level adjudication evidence rows:", len(rna_adjudication_files))
print("Selection eligibility rule: NOT_FROZEN")

Case-level adjudication evidence rows: 288
File-level adjudication evidence rows: 649
Selection eligibility rule: NOT_FROZEN


## PREFLIGHT-C — Source specimen provenance and technical QC *availability* (not adjudication)

**Question.** Does the **preserved acquisition-time GDC RNA metadata snapshot** contain sufficiently specific collection, spatial, preparation or comparable QC information to distinguish the already identified 649 RNA files from the 288 cases requiring review?

This stage first fingerprints the exact metadata snapshot already reported by Notebook 111 and examines **only those 649 audited file IDs**. For each file, additional fields are read **only from the case → sample → portion → analyte → aliquot path matching the exact three UUIDs already resolved in Notebook 111**. The step does not redo ancestry resolution or rescue the ambiguous records.

The audit distinguishes **key present**, **informative scalar value** (excluding common unknown/not-reported placeholders), and **multiple different values on matching hierarchical paths**. A GDC file creation date is **not** a specimen collection date. A sample submitter ID, portion number or concentration is **not** a QC score; generic `sample_type`, `tissue_type` or `tumor_descriptor` does **not** establish a tumor sampling region. An apparently uniform field does not prove identical tumor regions/timepoints.

**QC scope:** structural *availability of metadata fields* only. No processed STAR expression payloads, source QC reports, alignment logs or newly queried GDC information are inspected. Absent keys mean **not present in this audited snapshot**, *not* absent from the provider. No sample is selected or excluded; RNA file quality and relative quality remain unresolved.

In [21]:
# =============================================================================
# Read only the RNA acquisition-time metadata snapshot already frozen by 111
# =============================================================================

RNA_GDC_METADATA_PATH = (
    Paths.config / "manifests" / "cptac3" / "gdc_metadata_cptac3_rnaseq.json"
)
expected_rna_metadata_sha = audit_summary[
    "source_fingerprints"
]["rnaseq"]["metadata"]["sha256"]
if calculate_sha256(RNA_GDC_METADATA_PATH) != expected_rna_metadata_sha:
    raise ValueError("RNA acquisition-time GDC metadata differs from Notebook 111")

with RNA_GDC_METADATA_PATH.open("r", encoding="utf-8") as handle:
    rna_snapshot = json.load(handle)
if not isinstance(rna_snapshot, dict) or not isinstance(
    rna_snapshot.get("data"), dict
):
    raise ValueError("Unexpected RNA metadata snapshot envelope")
rna_metadata_hits = rna_snapshot["data"].get("hits")
if not isinstance(rna_metadata_hits, list):
    raise ValueError("RNA metadata snapshot has no data.hits list")

# The snapshot is already source-audited by 111: index only 800-B's 649 files.
review_file_ids = set(rna_adjudication_files["file_id"])
rna_c_hits = {}
for hit in rna_metadata_hits:
    file_uuid = hit.get("file_id") or hit.get("id")
    if file_uuid in review_file_ids:
        if file_uuid in rna_c_hits:
            raise ValueError("Duplicate file ID in reviewed RNA snapshot subset")
        rna_c_hits[file_uuid] = hit
if set(rna_c_hits) != review_file_ids:
    raise ValueError("Missing 800-B file IDs in frozen RNA metadata snapshot")

print("Exact reviewed GDC metadata files:", len(rna_c_hits))
print("RNA metadata snapshot SHA-256: PASS (Notebook 111 reference)")
print("No API query, source re-acquisition or STAR payload QC scan")

Exact reviewed GDC metadata files: 649
RNA metadata snapshot SHA-256: PASS (Notebook 111 reference)
No API query, source re-acquisition or STAR payload QC scan


In [22]:
# =============================================================================
# Locate exact already-audited triplets to expose additional source fields
# =============================================================================

def exact_source_nodes(hit, case_uuid, sample_uuid, aliquot_uuid):
    paths = []
    for case in hit.get("cases") or []:
        if case.get("case_id") != case_uuid:
            continue
        for sample in case.get("samples") or []:
            if sample.get("sample_id") != sample_uuid:
                continue
            for portion in sample.get("portions") or []:
                for analyte in portion.get("analytes") or []:
                    for aliquot in analyte.get("aliquots") or []:
                        if aliquot.get("aliquot_id") == aliquot_uuid:
                            paths.append({
                                "sample": sample,
                                "portion": portion,
                                "analyte": analyte,
                                "aliquot": aliquot,
                            })
    return paths

rna_c_node_maps = {}
rna_c_path_rows = []
for row in rna_adjudication_files.itertuples(index=False):
    hit = rna_c_hits[row.file_id]
    paths = exact_source_nodes(
        hit, row.case_uuid, row.sample_uuid, row.aliquot_uuid
    )
    if not paths:
        raise ValueError(
            "Notebook 111 audited case/sample/aliquot path not found for "
            + str(row.file_id)
        )
    rna_c_node_maps[row.file_id] = {
        "file": [hit],
        "analysis": [hit.get("analysis")] if isinstance(
            hit.get("analysis"), dict
        ) else [],
        **{
            scope: [entry[scope] for entry in paths]
            for scope in ("sample", "portion", "analyte", "aliquot")
        },
    }
    rna_c_path_rows.append({
        "file_id": row.file_id,
        "case_uuid": row.case_uuid,
        "sample_uuid": row.sample_uuid,
        "aliquot_uuid": row.aliquot_uuid,
        "exact_hierarchy_path_rows": len(paths),
    })

rna_c_exact_path_index = pd.DataFrame(rna_c_path_rows)
print("Exact previously audited triplets represented:",
      len(rna_c_exact_path_index))
print("Files with >1 exact hierarchy row (not distinct parents):",
      int(rna_c_exact_path_index["exact_hierarchy_path_rows"].gt(1).sum()))

Exact previously audited triplets represented: 649
Files with >1 exact hierarchy row (not distinct parents): 0


In [23]:
# =============================================================================
# Define conservative, testable source metadata availability domains
# =============================================================================

RNA_C_FIELDS = {
    "generic_sample_descriptors": [
        ("sample", "sample_type"), ("sample", "tissue_type"),
        ("sample", "tumor_descriptor"),
    ],
    "collection_timing": [
        ("sample", "days_to_collection"),
        ("sample", "days_to_sample_procurement"),
        ("sample", "time_between_clamping_and_freezing"),
        ("sample", "time_between_excision_and_freezing"),
    ],
    "sampling_location_detail": [
        ("sample", "tumor_region"), ("sample", "tumor_subsite"),
        ("sample", "sample_collection_site"), ("sample", "anatomic_site"),
        ("sample", "specimen_laterality"),
    ],
    "processing_provenance": [
        ("sample", "method_of_sample_procurement"),
        ("sample", "preservation_method"), ("sample", "is_ffpe"),
        ("portion", "is_ffpe"), ("portion", "portion_number"),
        ("portion", "weight"), ("analyte", "analyte_type"),
        ("analyte", "concentration"), ("aliquot", "concentration"),
    ],
    "identifiers_not_qc": [
        ("sample", "submitter_id"), ("portion", "submitter_id"),
        ("analyte", "submitter_id"), ("aliquot", "submitter_id"),
    ],
    "file_processing_dates_not_collection": [
        ("file", "created_datetime"), ("file", "updated_datetime"),
        ("analysis", "analysis_id"), ("analysis", "workflow_type"),
        ("analysis", "workflow_version"),
    ],
    "quantitative_qc_metadata_probes": [
        ("file", "total_reads"), ("file", "mapped_reads"),
        ("file", "average_read_length"), ("file", "average_base_quality"),
        ("analysis", "total_reads"), ("analysis", "mapped_reads"),
        ("analysis", "mapping_rate"), ("analysis", "duplication_rate"),
        ("analysis", "percent_aligned_reads"),
    ],
}

# Presence here means present in the preserved projection, not in all GDC data.
PLACEHOLDER_STRINGS = {
    "", "unknown", "not reported", "not available", "not applicable",
    "n/a", "na", "null", "none", "unspecified", "--",
}

def informative_scalar(value):
    if value is None or isinstance(value, (list, tuple, dict)):
        return False
    if isinstance(value, str):
        return value.strip().casefold() not in PLACEHOLDER_STRINGS
    if isinstance(value, (bool, int, float)):
        return bool(pd.notna(value))
    return False

def source_value_status(nodes, key):
    values = [
        item[key] for item in nodes
        if isinstance(item, dict) and key in item
    ]
    useful = {str(v).strip() for v in values if informative_scalar(v)}
    return bool(values), bool(useful), len(useful) > 1

all_probes = {
    (scope, key)
    for group in RNA_C_FIELDS.values()
    for scope, key in group
}
print("Explicit availability probes:", len(all_probes))
print("No thresholds for donor identity, specimen equivalence or quality")

Explicit availability probes: 39
No thresholds for donor identity, specimen equivalence or quality


In [24]:
# =============================================================================
# Inventory observed scalar snapshot keys and quantitative-QC probe coverage
# =============================================================================

observed_scalar_keys = set()
for node_map in rna_c_node_maps.values():
    for scope, nodes in node_map.items():
        for node in nodes:
            if isinstance(node, dict):
                observed_scalar_keys.update(
                    (scope, key) for key, value in node.items()
                    if not isinstance(value, (dict, list, tuple))
                )

# Inventory includes actual fields, plus explicit absent probes for auditability.
all_field_keys = sorted(all_probes | observed_scalar_keys)
rna_c_field_records = []
for scope, key in all_field_keys:
    n_key, n_informative, n_discordant = 0, 0, 0
    for nodes in rna_c_node_maps.values():
        present, informative, discordant = source_value_status(
            nodes[scope], key
        )
        n_key += int(present)
        n_informative += int(informative)
        n_discordant += int(discordant)
    assigned_domains = sorted(
        name for name, fields in RNA_C_FIELDS.items()
        if (scope, key) in fields
    )
    rna_c_field_records.append({
        "scope": scope, "field": key,
        "planned_domains": "|".join(assigned_domains),
        "observed_scalar_key": (scope, key) in observed_scalar_keys,
        "n_files_with_key": n_key,
        "n_files_with_informative_scalar": n_informative,
        "n_files_with_multiple_informative_values": n_discordant,
        "n_reviewed_files": len(rna_c_node_maps),
    })
rna_c_field_coverage = pd.DataFrame(rna_c_field_records)
display(
    rna_c_field_coverage.loc[
        rna_c_field_coverage["planned_domains"].ne("")
    ].sort_values(["planned_domains", "scope", "field"])
)
print(
    "Observed scalar fields not in explicit probe list:",
    len(observed_scalar_keys - all_probes)
)

,scope,field,planned_domains,observed_scalar_key,n_files_with_key,n_files_with_informative_scalar,n_files_with_multiple_informative_values,n_reviewed_files
34,sample,days_to_collection,collection_timing,False,0,0,0,649
35,sample,days_to_sample_procurement,collection_timing,False,0,0,0,649
44,sample,time_between_clamping_and_freezing,collection_timing,False,0,0,0,649
45,sample,time_between_excision_and_freezing,collection_timing,False,0,0,0,649
3,analysis,analysis_id,file_processing_dates_not_collection,False,0,0,0,649
9,analysis,workflow_type,file_processing_dates_not_collection,True,649,649,0,649
10,analysis,workflow_version,file_processing_dates_not_collection,True,649,649,0,649
17,file,created_datetime,file_processing_dates_not_collection,False,0,0,0,649
28,file,updated_datetime,file_processing_dates_not_collection,False,0,0,0,649
41,sample,sample_type,generic_sample_descriptors,True,649,649,0,649


Observed scalar fields not in explicit probe list: 11


In [25]:
# =============================================================================
# Flag usable specimen/QC metadata *availability* for each of 649 files
# =============================================================================

rna_c_file_rows = []
for row in rna_adjudication_files.itertuples(index=False):
    nodes = rna_c_node_maps[row.file_id]
    new_record = {
        "file_id": row.file_id,
        "case_uuid": row.case_uuid,
        "sample_uuid": row.sample_uuid,
        "aliquot_uuid": row.aliquot_uuid,
        "rna_multiplicity_class": row.rna_multiplicity_class,
    }
    for domain, probes in RNA_C_FIELDS.items():
        statuses = [
            source_value_status(nodes[scope], key)
            for scope, key in probes
        ]
        new_record["informative_" + domain] = any(
            informative for _, informative, _ in statuses
        )
        new_record["discordant_" + domain] = any(
            discordant for _, _, discordant in statuses
        )
        new_record["n_informative_fields_" + domain] = sum(
            informative for _, informative, _ in statuses
        )
    rna_c_file_rows.append(new_record)

rna_c_file_availability = pd.DataFrame(rna_c_file_rows).merge(
    rna_c_exact_path_index[["file_id", "exact_hierarchy_path_rows"]],
    on="file_id", validate="one_to_one"
).sort_values(["case_uuid", "sample_uuid", "aliquot_uuid", "file_id"])
rna_c_file_availability = rna_c_file_availability.reset_index(drop=True)

summary_fields = [
    "informative_" + name for name in RNA_C_FIELDS
]
display(
    rna_c_file_availability.groupby("rna_multiplicity_class")[
        summary_fields
    ].sum()
)
print("Non-informative placeholders are not counted as usable values")
print("Provider-level and processed STAR QC: NOT TESTED")

,informative_generic_sample_descriptors,informative_collection_timing,informative_sampling_location_detail,informative_processing_provenance,informative_identifiers_not_qc,informative_file_processing_dates_not_collection,informative_quantitative_qc_metadata_probes
rna_multiplicity_class,,,,,,,
MULTIPLE_SAMPLES_IN_CASE,619,0,0,0,619,619,0
ONE_SAMPLE_MULTIPLE_ALIQUOTS,30,0,0,0,30,30,0


Non-informative placeholders are not counted as usable values
Provider-level and processed STAR QC: NOT TESTED


In [26]:
# =============================================================================
# Summarize 288-case coverage without adjudicating within-case candidates
# =============================================================================

rna_c_case_coverage = (
    rna_c_file_availability.groupby("case_uuid")
    .agg(
        n_files=("file_id", "nunique"),
        n_exact_hierarchy_rows=("exact_hierarchy_path_rows", "sum"),
        **{
            "n_files_informative_" + name: (
                "informative_" + name, "sum"
            )
            for name in RNA_C_FIELDS
        },
        **{
            "n_files_discordant_" + name: (
                "discordant_" + name, "sum"
            )
            for name in RNA_C_FIELDS
        },
    )
    .reset_index()
    .merge(
        rna_review_cases[[
            "case_uuid", "rna_multiplicity_class", "n_files"
        ]].rename(columns={"n_files": "audited_n_files"}),
        on="case_uuid", validate="one_to_one"
    )
)
if (
    len(rna_c_case_coverage) != len(rna_review_cases)
    or not rna_c_case_coverage["n_files"].eq(
        rna_c_case_coverage["audited_n_files"]
    ).all()
):
    raise ValueError("800-C changed the source-snapshot review denominator")
rna_c_case_coverage["selection_policy_status"] = "NOT_FROZEN"
rna_c_case_coverage = rna_c_case_coverage.sort_values(
    "case_uuid", kind="stable"
).reset_index(drop=True)

display(
    rna_c_case_coverage.groupby("rna_multiplicity_class")
    .agg(
        cases=("case_uuid", "nunique"),
        with_any_collection_timing=(
            "n_files_informative_collection_timing", lambda v: int(v.gt(0).sum())
        ),
        with_all_collection_timing=(
            "n_files_informative_collection_timing",
            lambda v: int(v.eq(
                rna_c_case_coverage.loc[v.index, "n_files"]
            ).sum()),
        ),
        with_any_location_detail=(
            "n_files_informative_sampling_location_detail",
            lambda v: int(v.gt(0).sum())
        ),
        with_any_quantitative_qc_metadata=(
            "n_files_informative_quantitative_qc_metadata_probes",
            lambda v: int(v.gt(0).sum())
        ),
    )
)
print("All 288 cases remain unresolved for one-record-per-case selection")

,cases,with_any_collection_timing,with_all_collection_timing,with_any_location_detail,with_any_quantitative_qc_metadata
rna_multiplicity_class,,,,,
MULTIPLE_SAMPLES_IN_CASE,276,0,0,0,0
ONE_SAMPLE_MULTIPLE_ALIQUOTS,12,0,0,0,0


All 288 cases remain unresolved for one-record-per-case selection


In [27]:
# =============================================================================
# Enforce no-selection and scope guarantees for this availability-only stage
# =============================================================================

if (
    len(rna_c_file_availability) != len(rna_adjudication_files)
    or rna_c_file_availability["file_id"].duplicated().any()
    or rna_c_case_coverage["case_uuid"].duplicated().any()
    or rna_adjudication_files["candidate_selected"].any()
    or rna_adjudication_cases["selected_file_id"].notna().any()
    or rna_c_case_coverage["selection_policy_status"].ne("NOT_FROZEN").any()
):
    raise ValueError("800-C must not select or lose RNA source records")

print("PREFLIGHT-C source-metadata availability inventory: READY TO PERSIST")
print("Observed metadata availability is not evidence of relative sample QC")
print("Biological selection / program scoring: NOT PERFORMED")

PREFLIGHT-C source-metadata availability inventory: READY TO PERSIST
Observed metadata availability is not evidence of relative sample QC
Biological selection / program scoring: NOT PERFORMED


## Lexical identifier-overlap audit

Compare exact GDC case UUID and submitter strings against the frozen TCGA cohort. This is a **collision screen only**. Separate projects may assign different IDs to the same biological donor; therefore an absence of collisions is not evidence of documented donor independence. A provenance/crosswalk review remains a gate before confirmatory replication claims.

In [28]:
# =============================================================================
# Read TCGA source identities from the frozen Phase 2 sample mapping
# =============================================================================

rna_primary = candidate_primary.loc[
    candidate_primary["modality"].eq("rnaseq")
].copy()

tcga_mapping = pd.read_csv(frozen_paths["tcga_sample_mapping"], dtype="string")
tcga_required = {"rna_case_uuid", "case_submitter_id"}
if tcga_required - set(tcga_mapping.columns):
    raise KeyError("Frozen TCGA mapping lacks required case identifiers")

tcga_case_uuid_set = set(tcga_mapping["rna_case_uuid"].dropna())
tcga_submitter_set = set(tcga_mapping["case_submitter_id"].dropna())

print("Candidate CPTAC-3 RNA cases:", rna_primary["case_uuid"].nunique())
print("Donor independence: UNRESOLVED without source crosswalk")

Candidate CPTAC-3 RNA cases: 1370
Donor independence: UNRESOLVED without source crosswalk


In [29]:
# =============================================================================
# Create a case-level exact-string collision screen, not independence labels
# =============================================================================

identifier_collisions = (
    candidate_primary.groupby(["modality", "case_uuid"], dropna=False)
    .agg(
        n_submitter_ids=("case_submitter_id", "nunique"),
        submitter_id_values=(
            "case_submitter_id",
            lambda v: "|".join(sorted(set(v.dropna()))),
        ),
        primary_site_values=(
            "primary_site", lambda v: "|".join(sorted(set(v.dropna())))
        ),
        disease_type_values=(
            "disease_type", lambda v: "|".join(sorted(set(v.dropna())))
        ),
    )
    .reset_index()
)
identifier_collisions["tcga_case_uuid_collision"] = (
    identifier_collisions["case_uuid"].isin(tcga_case_uuid_set)
)
identifier_collisions["tcga_submitter_collision"] = (
    identifier_collisions["submitter_id_values"].map(
        lambda s: any(v in tcga_submitter_set for v in s.split("|") if v)
    )
)
identifier_collisions["donor_independence_status"] = (
    "UNRESOLVED_WITHOUT_EXTERNAL_PROVENANCE"
)
identifier_collisions["case_submitter_ambiguity"] = (
    identifier_collisions["n_submitter_ids"].gt(1)
)

display(
    identifier_collisions.groupby("modality")
    .agg(
        candidate_cases=("case_uuid", "nunique"),
        exact_uuid_collision_rows=("tcga_case_uuid_collision", "sum"),
        exact_submitter_collision_rows=("tcga_submitter_collision", "sum"),
    )
)

,candidate_cases,exact_uuid_collision_rows,exact_submitter_collision_rows
modality,,,
methylation,1336,0,0
rnaseq,1370,0,0


## Pilot technical gene-ID compatibility

The source library has already passed Notebook 111 file-integrity checks. A single deterministically selected *provisional primary RNA-seq* file is opened here **only** to inspect the GDC STAR Counts header and Ensembl identifier mapping to the 2,389-gene frozen universe.

This is a technical pilot, **not** a complete feature-coverage assessment, cohort normalization, or program test. A match after stripping Ensembl versions is reported only as a diagnostic; no version stripping, symbol fallback, missing-gene imputation or rescue is applied. Full per-file/lineage compatibility and thresholds remain open until reviewed.

In [30]:
# =============================================================================
# Select one deterministic RNA-seq technical pilot from the frozen source files
# =============================================================================

if rna_primary.empty:
    raise ValueError("Notebook 111 has no provisional primary RNA-seq candidates")
pilot_record = rna_primary.sort_values(
    ["file_id", "filename"], kind="stable"
).iloc[0]
pilot_file_uuid = pilot_record["file_id"]
pilot_path = (
    Paths.root / "data" / "raw" / "cptac3" / "rnaseq_star_counts"
    / pilot_file_uuid / pilot_record["filename"]
)
if not pilot_path.is_file():
    raise FileNotFoundError(f"Frozen Notebook 111 pilot payload is missing: {pilot_path}")

pilot_header = pd.read_csv(pilot_path, sep="\t", comment="#", nrows=0).columns.tolist()
print("Pilot GDC file:", pilot_file_uuid)
print("STAR Counts fields:", pilot_header)
if not {"gene_id", "unstranded"}.issubset(pilot_header):
    raise ValueError("Unexpected STAR Counts schema; no gene crosswalk attempted")

Pilot GDC file: 006f3179-76c6-4525-9560-2cf2a3e3e88f
STAR Counts fields: ['gene_id', 'gene_name', 'gene_type', 'unstranded', 'stranded_first', 'stranded_second', 'tpm_unstranded', 'fpkm_unstranded', 'fpkm_uq_unstranded']


In [31]:
# =============================================================================
# Audit exact/version-insensitive Ensembl ID coverage in the technical pilot
# =============================================================================

pilot_gene_ids = pd.read_csv(
    pilot_path, sep="\t", comment="#", usecols=["gene_id"], dtype="string"
)["gene_id"].dropna()
provider_ids = set(pilot_gene_ids.tolist())
provider_versions_stripped = pilot_gene_ids.str.replace(
    r"\.\d+$", "", regex=True
)
base_id_counts = provider_versions_stripped.value_counts()
ambiguous_base_ids = set(base_id_counts[base_id_counts.gt(1)].index)

gene_id_pilot = shared_genes[
    ["gene_symbol", "tumor_gene_id", "cell_line_gene_id"]
].copy()
gene_id_pilot["pilot_file_id"] = pilot_file_uuid
gene_id_pilot["exact_ensembl_match"] = (
    gene_id_pilot["tumor_gene_id"].isin(provider_ids)
)
gene_id_pilot["versionless_pilot_match_diagnostic"] = (
    gene_id_pilot["tumor_gene_id"].str.replace(r"\.\d+$", "", regex=True)
    .isin(set(provider_versions_stripped))
)
gene_id_pilot["versionless_pilot_id_ambiguous"] = (
    gene_id_pilot["tumor_gene_id"].str.replace(r"\.\d+$", "", regex=True)
    .isin(ambiguous_base_ids)
)
gene_id_pilot["mapping_applied"] = False

display(gene_id_pilot[
    ["exact_ensembl_match", "versionless_pilot_match_diagnostic",
     "versionless_pilot_id_ambiguous"]
].sum().to_frame(name="genes"))
print("Cohort-wide feature compatibility: NOT ASSESSED")

,genes
exact_ensembl_match,2389
versionless_pilot_match_diagnostic,2389
versionless_pilot_id_ambiguous,0


Cohort-wide feature compatibility: NOT ASSESSED


## Readiness and persistence

The preflight outputs remain local, provisional diagnostics under `data/interim/metadata`. They must not be registered as frozen biological validation evidence or used for Phase 9. The RNA-seq per-case selection and lineage crosswalk remain unresolved; project-specific donor independence remains unverified; the statistical Phase 8 analysis contract is not frozen.

New outputs receive basic shape/column validation after persistence, while Notebook 111's prior raw MD5 scan, upstream source inventories and frozen program-construction logic are not rerun.

In [32]:
# =============================================================================
# Assemble explicit readiness statuses and input provenance
# =============================================================================

input_provenance = {
    "111_summary": {
        "path": project_relative_path(AUDIT_SUMMARY_PATH),
        "sha256": calculate_sha256(AUDIT_SUMMARY_PATH),
    },
    "111_registry_proposal": {
        "path": project_relative_path(RAW_REGISTRY_PROPOSAL_PATH),
        "sha256": calculate_sha256(RAW_REGISTRY_PROPOSAL_PATH),
    },
}
input_provenance.update({
    "111_" + name: {
        "path": project_relative_path(path),
        "sha256": audit_summary["audit_artifacts"][name]["sha256"],
    }
    for name, path in audit_inputs.items()
})
input_provenance["gdc_rna_acquisition_metadata_for_c"] = {
    "path": project_relative_path(RNA_GDC_METADATA_PATH),
    "sha256": expected_rna_metadata_sha,
}
input_provenance.update({
    artifact_id: {
        "path": get_artifact(artifact_registry, artifact_id)["path"],
        "sha256": get_artifact(artifact_registry, artifact_id)["sha256"],
    }
    for artifact_id in FROZEN_ARTIFACT_IDS.values()
})

readiness_status = {
    "notebook111_source_integrity": "UPSTREAM_PASS_REUSED",
    "notebook111_local_handoff_identity": "PASS",
    "phase4_frozen_identity": "PASS",
    "provisional_primary_inventory": "AVAILABLE_NOT_COHORT_FROZEN",
    "one_record_per_case": "PENDING_POLICY",
    "rna_multiplicity_adjudication_evidence": "DESCRIBED_UNRESOLVED_NO_SELECTION",
    "rna_specimen_provenance_metadata": "SNAPSHOT_AVAILABILITY_AUDITED",
    "rna_quantitative_qc_metadata": "SNAPSHOT_PROBES_ONLY_NO_PAYLOAD_QC",
    "rna_one_record_per_case_selection": "NOT_FROZEN",
    "rna_case_multiplicity_classes": "DESCRIPTIVE_COMPLETE_NOT_RESOLVED",
    "rna_cross_file_ancestry_conflicts": (
        "REVIEW_REQUIRED" if rna_cases["cross_file_ancestry_conflict"].any()
        else "NONE_DETECTED_WITHIN_PROVISIONAL_PRIMARY_FILES"
    ),
    "harmonized_lineage_crosswalk": "NOT_ASSESSED",
    "donor_independence": "UNRESOLVED",
    "star_counts_gene_compatibility": "PILOT_ONLY",
    "full_transcriptomic_qc_and_coverage": "NOT_ASSESSED",
    "methylation_platform_and_sample_harmonization": "NOT_ASSESSED",
    "external_replication_analysis_contract": "NOT_FROZEN",
    "biological_validation_authorized": False,
}
display(pd.DataFrame(
    [{"gate": gate, "status": status} for gate, status in readiness_status.items()]
))

,gate,status
0,notebook111_source_integrity,UPSTREAM_PASS_REUSED
1,notebook111_local_handoff_identity,PASS
2,phase4_frozen_identity,PASS
3,provisional_primary_inventory,AVAILABLE_NOT_COHORT_FROZEN
4,one_record_per_case,PENDING_POLICY
5,rna_multiplicity_adjudication_evidence,DESCRIBED_UNRESOLVED_NO_SELECTION
6,rna_specimen_provenance_metadata,SNAPSHOT_AVAILABILITY_AUDITED
7,rna_quantitative_qc_metadata,SNAPSHOT_PROBES_ONLY_NO_PAYLOAD_QC
8,rna_one_record_per_case_selection,NOT_FROZEN
9,rna_case_multiplicity_classes,DESCRIPTIVE_COMPLETE_NOT_RESOLVED


In [33]:
# =============================================================================
# Persist new technical inventories; leave registries and raw data untouched
# =============================================================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_tables = {
    "lineage_inventory": lineage_inventory,
    "case_multiplicity": case_multiplicity,
    "identifier_collisions": identifier_collisions,
    "gene_id_pilot": gene_id_pilot,
    "rna_adjudication_case_evidence": rna_adjudication_cases,
    "rna_adjudication_file_evidence": rna_adjudication_files,
    "rna_c_field_coverage": rna_c_field_coverage,
    "rna_c_file_availability": rna_c_file_availability,
    "rna_c_case_coverage": rna_c_case_coverage,
}
for name, frame in output_tables.items():
    frame.to_csv(OUTPUT_PATHS[name], index=False)
    print(name, "→", project_relative_path(OUTPUT_PATHS[name]), "rows:", len(frame))

lineage_inventory → data/interim/metadata/800_preflight_a_candidate_lineage_inventory.csv rows: 61
case_multiplicity → data/interim/metadata/800_preflight_a_case_multiplicity.csv rows: 2706
identifier_collisions → data/interim/metadata/800_preflight_a_identifier_collisions.csv rows: 2706
gene_id_pilot → data/interim/metadata/800_preflight_a_gene_id_pilot.csv rows: 2389
rna_adjudication_case_evidence → data/interim/metadata/800_preflight_b_rna_case_evidence.csv rows: 288
rna_adjudication_file_evidence → data/interim/metadata/800_preflight_b_rna_file_evidence.csv rows: 649
rna_c_field_coverage → data/interim/metadata/800_preflight_c_rna_field_coverage.csv rows: 50
rna_c_file_availability → data/interim/metadata/800_preflight_c_rna_file_availability.csv rows: 649
rna_c_case_coverage → data/interim/metadata/800_preflight_c_rna_case_coverage.csv rows: 288


In [34]:
# =============================================================================
# Validate newly persisted technical inventories, not all upstream schemas
# =============================================================================

for name, frame in output_tables.items():
    reloaded = pd.read_csv(OUTPUT_PATHS[name], dtype="string")
    if len(reloaded) != len(frame) or list(reloaded.columns) != list(frame.columns):
        raise ValueError(f"New preflight output is inconsistent: {name}")

print("New technical inventory persistence: PASS")

New technical inventory persistence: PASS


In [35]:
# =============================================================================
# Write the non-frozen readiness record with provenance and output fingerprints
# =============================================================================

readiness_record = {
    "notebook": "800_external_bulk_cohorts",
    "stage": "PREFLIGHT_A_B_C_METADATA_AVAILABILITY_ONLY",
    "environment": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
    },
    "upstream_111_snapshot": audit_summary["source_fingerprints"],
    "inputs": input_provenance,
    "raw_registry_cptac3_present": cptac3_registered,
    "source_count_context": audit_summary["n_files_by_modality"],
    "program_gene_universe_n": len(shared_genes),
    "rna_case_multiplicity_class_counts": {
        str(row.rna_multiplicity_class): int(row.n_cases)
        for row in rna_class_summary.itertuples(index=False)
    },
    "rna_preflight_b_evidence_class_counts": {
        str(k): int(v) for k, v in
        rna_adjudication_cases["adjudication_evidence_class"]
        .value_counts(dropna=False).items()
    },
    "rna_preflight_b_case_count": len(rna_adjudication_cases),
    "rna_preflight_b_descriptor_discordance_cases": int(
        rna_adjudication_cases["source_descriptor_discordance"].sum()
    ),
    "rna_preflight_b_descriptor_incomplete_cases": int(
        rna_adjudication_cases["source_descriptor_incomplete"].sum()
    ),
    "rna_preflight_b_selection_performed": False,
    "rna_preflight_c_reviewed_files": len(rna_c_file_availability),
    "rna_preflight_c_reviewed_cases": len(rna_c_case_coverage),
    "rna_preflight_c_quantitative_qc_snapshot_files": int(
        rna_c_file_availability[
            "informative_quantitative_qc_metadata_probes"
        ].sum()
    ),
    "rna_preflight_c_selection_performed": False,
    "pilot_file_uuid": str(pilot_file_uuid),
    "pilot_star_counts_fields": pilot_header,
    "readiness": readiness_status,
    "cohort_frozen": False,
    "program_scores_computed": False,
    "external_replication_test_performed": False,
    "outputs": {
        name: {
            "path": project_relative_path(OUTPUT_PATHS[name]),
            "rows": len(output_tables[name]),
            "sha256": calculate_sha256(OUTPUT_PATHS[name]),
        }
        for name in output_tables
    },
}
with OUTPUT_PATHS["readiness"].open("w", encoding="utf-8") as handle:
    json.dump(readiness_record, handle, indent=2, ensure_ascii=False)
    handle.write("\n")
with OUTPUT_PATHS["readiness"].open("r", encoding="utf-8") as handle:
    persisted_readiness = json.load(handle)
if persisted_readiness["readiness"] != readiness_status:
    raise ValueError("Persisted readiness differs from the in-memory record")
print("Readiness saved:", project_relative_path(OUTPUT_PATHS["readiness"]))

Readiness saved: data/interim/metadata/800_preflight_a_readiness.json


## PREFLIGHT-A checkpoint — review required before Phase 8 replication

**Technical evidence only.** This notebook records whether Notebook 111's local derived handoffs and frozen Phase 4 inputs are readable and unchanged, and prepares detailed diagnostics of the provisional CPTAC-3 candidate cohort.

**Still open:** external donor independence beyond lexical ID collisions; lineage-specific attribution of sample-ancestry attrition; primary case-level selection; per-file STAR gene coverage and metadata/workflow QC; normalization/reference scaling; independent program-replication estimand and corresponding null; sample- and platform-aware methylation context; prospective multiplicity and inference policy.

**Next decision:** review the new technical tables and the Phase 8 preflight draft before freezing any cohort or analysis contract. A successful Run All here does *not* authorize biological replication testing or prove cohort readiness. Positive, negative and not-evaluable results must later be reported under the same frozen inferential design.

**PREFLIGHT-C checkpoint — snapshot availability only.** For 649 already-reviewed RNA files from 288 cases, the original Notebook 111 acquisition-time GDC metadata snapshot is reused by exact frozen file UUID and already audited case/sample/aliquot links. The new field-, file- and case-level availability diagnostics distinguish missing keys, placeholder/non-informative values, and conflicting values on exact matching hierarchy paths. This does not audit all GDC metadata endpoints, raw sequencing QC, processed STAR QC, tumor spatial/timepoint equivalence or relative quality. Absence from this snapshot is not provider-wide absence. No specimens are selected, weighted, averaged or excluded.

**PREFLIGHT-B checkpoint — no file adjudication yet.** The 288 RNA-seq multiplicity cases are reviewed only using already audited source-file and biospecimen descriptors. New per-case and per-file evidence inventories preserve source UUIDs, missing/discordant metadata and the absence of a defensible technical QC selection rule. Neither a case flagged for discordance nor a technically simple case is automatically excluded or included. Final one-record-per-case selection, replicate equivalence, raw RNA-seq QC and specimen provenance remain unassessed. No output belongs to frozen validation results.

**Additional case-level multiplicity audit (outcome-blind).** The RNA-seq provisional case inventory now reports mutually exclusive identifier-structure classes, including cross-file ancestry conflicts, without requiring re-execution of Notebook 111, promoting any file, or changing `NOT_FROZEN` selection status. The case-multiplicity CSV gains class/contradiction fields; the readiness JSON records class counts. Actual classes must be reviewed after local Run All; no category is a final analytic-eligibility decision.